<a href="https://colab.research.google.com/github/Nilufar-Komil/Applications-of-the-Generalized-Stiefel-Manifold-in-NN/blob/small_run/Vit_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:

# -*- coding: utf-8 -*-

# ------------------------------------------------------------
# ViT-Tiny/4: Fixed-Rank Generalized Stiefel
# Imports and device setup
# ------------------------------------------------------------

import os
import sys
import math
import random
from typing import Optional

sys.path.append(os.getcwd())

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from torchvision import transforms
from torch.optim import SGD, Adam
from torch.utils.data import Dataset, DataLoader, random_split

# Numerical computations
import numpy as np

# Data processing
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Device setup
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device available:", device)


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



Device available: cpu


In [33]:



# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: CIFAR-10 Dataset
# ------------------------------------------------------------

# CIFAR-10 normalization
# Same as vit_sgd and vit_stiefel

cifar10_mean = (125.3 / 255.0, 123.0 / 255.0, 113.9 / 255.0)
cifar10_std = (63.0 / 255.0, 62.1 / 255.0, 66.7 / 255.0)

# ------------------------------------------------------------
# Training transforms
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Pad(4, padding_mode="reflect"),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# ------------------------------------------------------------
# Validation and test transforms
# ------------------------------------------------------------

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# ------------------------------------------------------------
# CIFAR-10 datasets
# ------------------------------------------------------------

# Augmented training dataset
full_train_dataset_aug = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

# Clean training dataset for validation
full_train_dataset_clean = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=test_transform,
)

# Official CIFAR-10 test dataset
test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

# ------------------------------------------------------------
# Reproducible 80/20 training-validation split
# ------------------------------------------------------------

train_size = int(0.8 * len(full_train_dataset_aug))
val_size = len(full_train_dataset_aug) - train_size

split_seed = 42

train_dataset, _ = random_split(
    full_train_dataset_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

_, val_dataset = random_split(
    full_train_dataset_clean,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

# ------------------------------------------------------------
# Dataset summary
# ------------------------------------------------------------

print("Full train size:", len(full_train_dataset_aug))
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))


Full train size: 50000
Train size: 40000
Val size: 10000
Test size: 10000


In [3]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: CIFAR-10 DataLoaders
# ------------------------------------------------------------

batch_size = 128

# Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

# Validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

# Test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print("Batch size:", batch_size)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


Batch size: 128
Training batches: 313
Validation batches: 79
Test batches: 79


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [4]:

# ------------------------------------------------------------
# ViT-Tiny/4: Product Generalized Stiefel Projection
# ------------------------------------------------------------

import torch
import torch.nn as nn


class ProductGStiefelAttentionProjection(nn.Module):

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        diffeo,
        bias: bool = True,
    ):
        super().__init__()

        if embed_dim % num_heads != 0:
            raise ValueError(
                "embed_dim must be divisible by num_heads"
            )

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.diffeo = diffeo

        self.n = embed_dim
        self.p = self.head_dim
        self.rank = diffeo.rank

        # Feasibility condition
        if not (self.p <= self.rank <= self.n):
            raise ValueError(
                f"Expected p <= rank <= n, "
                f"got p={self.p}, "
                f"rank={self.rank}, n={self.n}"
            )

        # ----------------------------------------------------
        # Initialize product-manifold coordinates
        # ----------------------------------------------------

        X_list = []
        Z_list = []

        for h in range(num_heads):

            X_std, Z_std = (
                diffeo.random_product_point(self.p)
            )

            # X_std: (rank, p)
            # Z_std: (n-rank, p)

            X_list.append(X_std.contiguous())
            Z_list.append(Z_std.contiguous())

        # X: (num_heads, rank, p)
        # Z: (num_heads, n-rank, p)

        self.X = nn.Parameter(
            torch.stack(X_list, dim=0)
        )

        self.Z = nn.Parameter(
            torch.stack(Z_list, dim=0)
        )

        # ----------------------------------------------------
        # Bias for each attention head
        # ----------------------------------------------------

        if bias:
            self.bias = nn.Parameter(
                torch.zeros(num_heads, self.p)
            )
        else:
            self.register_parameter("bias", None)

    # --------------------------------------------------------
    # Reconstruct generalized Stiefel weights
    # --------------------------------------------------------

    def reconstruct_weight(self) -> torch.Tensor:

        W_list = []

        for h in range(self.num_heads):

            X_std = self.X[h]
            Z_std = self.Z[h]

            # W_h: (embed_dim, head_dim)
            W_h = self.diffeo.phi_inverse(
                X_std, Z_std
            )

            if not isinstance(W_h, torch.Tensor):
                raise TypeError(
                    "phi_inverse must return a torch.Tensor"
                )

            W_list.append(W_h)

        # W: (num_heads, embed_dim, head_dim)
        W = torch.stack(W_list, dim=0)

        return W

    # --------------------------------------------------------
    # Forward projection
    # --------------------------------------------------------

    def forward(self, x: torch.Tensor):

        # x: (batch, tokens, embed_dim)

        W = self.reconstruct_weight()

        # Project each attention head
        # out: (batch, heads, tokens, head_dim)

        out = torch.einsum(
            "bnd,hdp->bhnp",
            x,
            W
        )

        if self.bias is not None:
            out = out + self.bias[None, :, None, :]

        return out


In [5]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Transformer Basic Block
# ------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Multi-Head Self-Attention
# Q and K: Product Generalized Stiefel
# V and output projection: Euclidean
# ------------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        q_proj_layer,
        k_proj_layer,
        embed_dim=192,
        num_heads=3,
        dropout=0.0,
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Fixed-rank generalized Stiefel projections
        self.q_proj = q_proj_layer
        self.k_proj = k_proj_layer

        # Euclidean V projection
        shape = (num_heads, embed_dim, self.head_dim)

        self.W_v = nn.Parameter(torch.empty(shape))

        self.b_v = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )

        # Euclidean output projection
        self.proj = nn.Linear(embed_dim, embed_dim)

        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):

        # Initialize only Euclidean parameters
        for h in range(self.num_heads):
            nn.init.orthogonal_(self.W_v[h])

        nn.init.zeros_(self.b_v)

        nn.init.orthogonal_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

        # Q/K product coordinates are initialized
        # inside ProductGStiefelAttentionProjection

    def forward(self, x):

        # x: (batch, tokens, embed_dim)
        B, N, C = x.shape

        # Generalized Stiefel Q and K
        # (batch, heads, tokens, head_dim)
        q = self.q_proj(x)
        k = self.k_proj(x)

        # Euclidean V projection
        v = torch.einsum(
            "bnd,hdp->bhnp",
            x,
            self.W_v
        )

        v = v + self.b_v[None, :, None, :]

        # Attention scores
        attn = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attn = torch.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        out = attn @ v

        # Combine attention heads
        out = out.transpose(1, 2).reshape(B, N, C)

        out = self.proj(out)
        out = self.proj_drop(out)

        return out


# ------------------------------------------------------------
# Transformer MLP
# Same as vit_sgd and vit_stiefel
# ------------------------------------------------------------

class TransformerMLP(nn.Module):

    def __init__(
        self,
        embed_dim=192,
        mlp_ratio=4.0,
        dropout=0.0,
    ):
        super().__init__()

        hidden_dim = int(embed_dim * mlp_ratio)

        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):

        nn.init.orthogonal_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

        nn.init.orthogonal_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):

        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.drop2(x)

        return x


# ------------------------------------------------------------
# ViT Basic Block
# Replaces WRNBasicBlock
# ------------------------------------------------------------

class ViTBasicBlock(nn.Module):

    def __init__(
        self,
        q_proj_layer,
        k_proj_layer,
        embed_dim=192,
        num_heads=3,
        mlp_ratio=4.0,
        dropout=0.0,
    ):
        super().__init__()

        # First LayerNorm
        self.norm1 = nn.LayerNorm(embed_dim)

        # Multi-head attention with fixed-rank Q/K
        self.attn = MultiHeadSelfAttention(
            q_proj_layer=q_proj_layer,
            k_proj_layer=k_proj_layer,
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

        # Second LayerNorm
        self.norm2 = nn.LayerNorm(embed_dim)

        # Euclidean MLP
        self.mlp = TransformerMLP(
            embed_dim=embed_dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.reset_parameters()

    def reset_parameters(self):

        nn.init.ones_(self.norm1.weight)
        nn.init.zeros_(self.norm1.bias)

        nn.init.ones_(self.norm2.weight)
        nn.init.zeros_(self.norm2.bias)

    def forward(self, x):

        # Attention + residual connection
        x = x + self.attn(self.norm1(x))

        # MLP + residual connection
        x = x + self.mlp(self.norm2(x))

        return x


In [6]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Transformer Group
# Replaces WRNGroup
# ------------------------------------------------------------

class ViTGroup(nn.Module):

    def __init__(
        self,
        num_blocks,
        embed_dim=192,
        num_heads=3,
        mlp_ratio=4.0,
        dropout=0.0,
        block_builder=None,
        group_index=None,
    ):
        super().__init__()

        blocks = []

        if block_builder is None:
            raise ValueError(
                "Fixed-rank ViT requires a block_builder "
                "to construct the generalized Stiefel "
                "Q/K projection layers."
            )

        for i in range(num_blocks):

            block = block_builder(
                group_index=group_index,
                block_index=i,
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
            )

            if not isinstance(block, nn.Module):
                raise TypeError(
                    "block_builder must return an nn.Module"
                )

            blocks.append(block)

        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


In [7]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank
# Replaces WideResNet
# ------------------------------------------------------------

class ViTTiny4(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        depth=12,
        embed_dim=192,
        num_heads=3,
        num_classes=10,
        mlp_ratio=4.0,
        dropout=0.0,
        patch_embed_layer=None,
        block_builder=None,
    ):
        super().__init__()

        assert img_size % patch_size == 0
        assert depth % 3 == 0
        assert embed_dim % num_heads == 0

        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.depth = depth

        self.num_patches = (
            img_size // patch_size
        ) ** 2

        # ----------------------------------------------------
        # Patch embedding
        # (B, 3, 32, 32) -> (B, 192, 8, 8)
        # ----------------------------------------------------

        self.patch_embed = (
            patch_embed_layer
            if patch_embed_layer is not None
            else nn.Conv2d(
                in_channels=3,
                out_channels=embed_dim,
                kernel_size=patch_size,
                stride=patch_size,
                bias=True,
            )
        )

        # ----------------------------------------------------
        # Classification token
        # ----------------------------------------------------

        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )

        # ----------------------------------------------------
        # Learnable positional embeddings
        # 64 patches + 1 CLS token
        # ----------------------------------------------------

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                self.num_patches + 1,
                embed_dim,
            )
        )

        self.pos_drop = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Transformer encoder groups
        # 12 blocks = 3 groups x 4 blocks
        # ----------------------------------------------------

        blocks_per_group = depth // 3

        self.group0 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            block_builder=block_builder,
            group_index=0,
        )

        self.group1 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            block_builder=block_builder,
            group_index=1,
        )

        self.group2 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            block_builder=block_builder,
            group_index=2,
        )

        # ----------------------------------------------------
        # Final normalization
        # ----------------------------------------------------

        self.norm = nn.LayerNorm(embed_dim)

        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        self.head = nn.Linear(
            embed_dim,
            num_classes,
        )

        self.reset_parameters()

    # --------------------------------------------------------
    # Parameter initialization
    # --------------------------------------------------------

    def reset_parameters(self):

        # Initialize ordinary patch embedding
        if isinstance(self.patch_embed, nn.Conv2d):

            nn.init.orthogonal_(
                self.patch_embed.weight
            )

            if self.patch_embed.bias is not None:
                nn.init.zeros_(
                    self.patch_embed.bias
                )

        # Classification token
        nn.init.normal_(
            self.cls_token,
            mean=0.0,
            std=0.02,
        )

        # Positional embeddings
        nn.init.normal_(
            self.pos_embed,
            mean=0.0,
            std=0.02,
        )

        # Final LayerNorm
        nn.init.ones_(self.norm.weight)
        nn.init.zeros_(self.norm.bias)

        # Classification head
        nn.init.normal_(
            self.head.weight,
            mean=0.0,
            std=2 / (
                self.head.in_features ** 0.5
            ),
        )

        nn.init.zeros_(self.head.bias)

        # Q/K product-manifold coordinates are
        # initialized by their respective layers.

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    def forward(self, x):

        # Patch embedding
        # (B, 3, 32, 32) -> (B, 192, 8, 8)

        x = self.patch_embed(x)

        # Flatten spatial dimensions
        # (B, 192, 8, 8) -> (B, 64, 192)

        x = x.flatten(2).transpose(1, 2)

        # ----------------------------------------------------
        # Add CLS token
        # ----------------------------------------------------

        cls_token = self.cls_token.expand(
            x.shape[0],
            -1,
            -1,
        )

        x = torch.cat(
            (cls_token, x),
            dim=1,
        )

        # ----------------------------------------------------
        # Add positional embeddings
        # (B, 65, 192)
        # ----------------------------------------------------

        x = self.pos_drop(
            x + self.pos_embed
        )

        # ----------------------------------------------------
        # Transformer groups
        # Q/K reconstructed through product manifold
        # ----------------------------------------------------

        x = self.group0(x)
        x = self.group1(x)
        x = self.group2(x)

        # ----------------------------------------------------
        # Final LayerNorm
        # ----------------------------------------------------

        x = self.norm(x)

        # Extract CLS token
        x = x[:, 0]

        # Classification
        x = self.head(x)

        return x


In [8]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Generalized Stiefel feasibility helper
# ------------------------------------------------------------

def is_feasible_gstiefel_attention(
    embed_dim,
    num_heads,
    rank,
):
    if embed_dim % num_heads != 0:
        return False

    n = embed_dim
    p = embed_dim // num_heads

    return (p <= n) and (p <= rank <= n)


In [9]:

embed_dim = 192
num_heads = 3

for rank in [32, 64, 96, 128, 192]:
    feasible = is_feasible_gstiefel_attention(
        embed_dim=embed_dim,
        num_heads=num_heads,
        rank=rank,
    )
    print(f"Rank {rank}: feasible = {feasible}")


Rank 32: feasible = False
Rank 64: feasible = True
Rank 96: feasible = True
Rank 128: feasible = True
Rank 192: feasible = True


In [10]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Block Builder
# Replaces the WRN make_block_builder
# ------------------------------------------------------------

def make_block_builder(diffeo_factory):
    """
    diffeo_factory(
        embed_dim,
        head_dim,
        group_index,
        block_index,
        proj_name,
    )

    Returns a fixed-rank generalized Stiefel
    diffeomorphism for each Q/K attention projection.
    """

    def build_block(
        group_index,
        block_index,
        embed_dim=192,
        num_heads=3,
        mlp_ratio=4.0,
        dropout=0.0,
    ):

        head_dim = embed_dim // num_heads

        # ----------------------------------------------------
        # Q projection: generalized Stiefel
        # ----------------------------------------------------

        diffeo_q = diffeo_factory(
            embed_dim=embed_dim,
            head_dim=head_dim,
            group_index=group_index,
            block_index=block_index,
            proj_name="q",
        )

        if not is_feasible_gstiefel_attention(
            embed_dim,
            num_heads,
            diffeo_q.rank,
        ):
            raise ValueError(
                f"Infeasible Q projection at "
                f"group{group_index}.block{block_index}: "
                f"rank={diffeo_q.rank}"
            )

        q_proj_layer = ProductGStiefelAttentionProjection(
            embed_dim=embed_dim,
            num_heads=num_heads,
            diffeo=diffeo_q,
            bias=True,
        )

        # ----------------------------------------------------
        # K projection: generalized Stiefel
        # ----------------------------------------------------

        diffeo_k = diffeo_factory(
            embed_dim=embed_dim,
            head_dim=head_dim,
            group_index=group_index,
            block_index=block_index,
            proj_name="k",
        )

        if not is_feasible_gstiefel_attention(
            embed_dim,
            num_heads,
            diffeo_k.rank,
        ):
            raise ValueError(
                f"Infeasible K projection at "
                f"group{group_index}.block{block_index}: "
                f"rank={diffeo_k.rank}"
            )

        k_proj_layer = ProductGStiefelAttentionProjection(
            embed_dim=embed_dim,
            num_heads=num_heads,
            diffeo=diffeo_k,
            bias=True,
        )

        # ----------------------------------------------------
        # Construct ViT Transformer block
        # ----------------------------------------------------

        return ViTBasicBlock(
            q_proj_layer=q_proj_layer,
            k_proj_layer=k_proj_layer,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

    return build_block


In [11]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Patch Embedding Layer
# Replaces WRN make_stem_layer
# ------------------------------------------------------------

def make_patch_embed_layer(
    embed_dim=192,
    patch_size=4,
):
    return nn.Conv2d(
        in_channels=3,
        out_channels=embed_dim,
        kernel_size=patch_size,
        stride=patch_size,
        bias=True,
    )


In [13]:

import torch

def sym(A: torch.Tensor) -> torch.Tensor:
    """
    Compute the symmetric part of a matrix.

    Supports:
        A: (n, n)
        A: (num_heads, n, n)
    """
    return 0.5 * (A + A.transpose(-1, -2))


In [14]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Generalized Stiefel Diffeomorphism
# ------------------------------------------------------------

import torch
import torch.nn as nn


class GeneralizedStiefelDiffeomorphismTorch(nn.Module):

    def __init__(
        self,
        B: torch.Tensor,
        tol: float = 1e-10,
    ):
        super().__init__()

        if B.ndim != 2 or B.shape[0] != B.shape[1]:
            raise ValueError("B must be a square matrix.")

        self.tol = tol
        self.n = B.shape[0]

        # ----------------------------------------------------
        # Diagonal PSD metric
        # ----------------------------------------------------

        d = torch.diagonal(B)

        # This implementation assumes diagonal B
        if not torch.allclose(
            B,
            torch.diag(d),
            atol=tol,
            rtol=0.0,
        ):
            raise ValueError("B must be diagonal.")

        if torch.any(d < -tol):
            raise ValueError(
                "B must be positive semidefinite."
            )

        d = torch.clamp(d, min=0.0)

        # Zeros first, positive eigenvalues last
        self.rank = int((d > tol).sum().item())
        self.zero_count = self.n - self.rank

        if torch.any(d[:self.zero_count] > tol):
            raise ValueError(
                "Zero eigenvalues must come first."
            )

        if torch.any(d[self.zero_count:] <= tol):
            raise ValueError(
                "Positive eigenvalues must come last."
            )

        # Register B as a fixed, non-trainable buffer
        # It will move automatically with model.to(device)
        self.register_buffer(
            "B",
            torch.diag(d)
        )

        # ----------------------------------------------------
        # Spectral factors
        # ----------------------------------------------------

        if self.rank > 0:

            positive_d = d[self.zero_count:]

            lambda_half_inv = (
                1.0 / torch.sqrt(positive_d)
            )

            lambda_half = torch.sqrt(positive_d)

        else:

            lambda_half_inv = d.new_empty(0)
            lambda_half = d.new_empty(0)

        self.register_buffer(
            "lambda_half_inv_vec",
            lambda_half_inv
        )

        self.register_buffer(
            "lambda_half_vec",
            lambda_half
        )

    # --------------------------------------------------------
    # Check generalized Stiefel constraint
    # --------------------------------------------------------

    @torch.no_grad()
    def in_generalized_stiefel(
        self,
        W: torch.Tensor,
    ) -> bool:

        if W.ndim != 2 or W.shape[0] != self.n:
            return False

        p = W.shape[1]

        # Efficient computation for diagonal B
        d = torch.diagonal(self.B)

        gram = W.T @ (d[:, None] * W)

        I = torch.eye(
            p,
            device=W.device,
            dtype=W.dtype,
        )

        error = torch.linalg.matrix_norm(
            gram - I,
            ord="fro",
        )

        # Account for float32 roundoff
        numerical_tol = (
            10 * torch.finfo(W.dtype).eps * p
        )

        threshold = max(
            1e2 * self.tol,
            numerical_tol,
        )

        return bool(error.item() <= threshold)

    # --------------------------------------------------------
    # Forward diffeomorphism
    # W -> (X, Z)
    # --------------------------------------------------------

    def phi(
        self,
        W: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        if W.ndim != 2 or W.shape[0] != self.n:
            raise ValueError(
                f"W must have shape ({self.n}, p)."
            )

        # Null-space coordinates
        Z = W[:self.zero_count, :]

        # Positive-eigenvalue coordinates
        W1 = W[self.zero_count:, :]

        X = (
            self.lambda_half_vec[:, None] * W1
        )

        return X, Z

    # --------------------------------------------------------
    # Inverse diffeomorphism
    # (X, Z) -> W
    # --------------------------------------------------------

    def phi_inverse(
        self,
        X: torch.Tensor,
        Z: torch.Tensor,
    ) -> torch.Tensor:

        if X.ndim != 2 or Z.ndim != 2:
            raise ValueError(
                "X and Z must be 2D matrices."
            )

        if X.shape[0] != self.rank:
            raise ValueError(
                f"X must have {self.rank} rows."
            )

        if Z.shape[0] != self.zero_count:
            raise ValueError(
                f"Z must have {self.zero_count} rows."
            )

        if X.shape[1] != Z.shape[1]:
            raise ValueError(
                "X and Z must have the same "
                "number of columns."
            )

        # Positive-eigenvalue component
        W1 = (
            self.lambda_half_inv_vec[:, None] * X
        )

        # Combine null and positive components
        # torch.cat preserves autograd
        W = torch.cat(
            (Z, W1),
            dim=0,
        )

        return W

    # --------------------------------------------------------
    # Random product-manifold point
    # --------------------------------------------------------

    @torch.no_grad()
    def random_product_point(
        self,
        p: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        if p > self.rank:
            raise ValueError(
                f"Cannot create a Stiefel point "
                f"with p={p} when rank(B)={self.rank}."
            )

        A = torch.randn(
            self.rank,
            p,
            device=self.B.device,
            dtype=self.B.dtype,
        )

        Qx, Rx = torch.linalg.qr(
            A,
            mode="reduced",
        )

        # Sign correction
        d = torch.diagonal(Rx)

        s = torch.where(
            d < 0,
            -torch.ones_like(d),
            torch.ones_like(d),
        )

        X = Qx * s.unsqueeze(0)

        Z = torch.zeros(
            self.zero_count,
            p,
            device=self.B.device,
            dtype=self.B.dtype,
        )

        return X, Z

    # --------------------------------------------------------
    # Initialize product coordinates from W
    # --------------------------------------------------------

    def init_from_W(
        self,
        W: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:

        if not self.in_generalized_stiefel(W):
            raise ValueError(
                "W does not satisfy W^T B W = I "
                "within numerical tolerance."
            )

        return self.phi(W)

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    def print_summary(self) -> None:

        print(f"n = {self.n}")
        print(f"rank(B) = {self.rank}")
        print(f"zero_count = {self.zero_count}")
        print(f"device = {self.B.device}")
        print(f"dtype = {self.B.dtype}")


In [16]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Metric B and Diffeomorphism Factory
# ------------------------------------------------------------

import math
import torch

dtype = torch.float32

# ------------------------------------------------------------
# Fixed rank configuration
# ------------------------------------------------------------

# Original WRN setting: rank(B) = n
FIXED_RANK = 192

# For rank-deficient experiments, examples:
# FIXED_RANK = 96
# FIXED_RANK = 64


# ------------------------------------------------------------
# Construct diagonal PSD metric B
# ------------------------------------------------------------

def make_B_for_attention(
    n,
    p,
    device,
    dtype=torch.float32,
    target_rank=None,
):
    """
    Construct a diagonal positive semidefinite B
    with fixed rank.

    n: input embedding dimension
    p: output dimension per attention head
    """

    # Preserve original WRN behavior: rank(B) = n
    if target_rank is None:
        target_rank = n

    if not (p <= target_rank <= n):
        raise ValueError(
            f"Expected p <= rank <= n, "
            f"got p={p}, rank={target_rank}, n={n}"
        )

    zero_count = n - target_rank

    # Same eigenvalue initialization as WRN
    d = torch.linspace(
        0.1,
        2.5,
        n,
        device=device,
        dtype=dtype,
    )

    # Zero eigenvalues first
    d[:zero_count] = 0.0

    # Diagonal PSD metric
    B = torch.diag(d)

    return B, target_rank


# ------------------------------------------------------------
# Diffeomorphism Factory for Q/K projections
# ------------------------------------------------------------

def diffeo_factory(
    embed_dim,
    head_dim,
    group_index,
    block_index,
    proj_name,
):
    """
    Create a generalized Stiefel diffeomorphism
    for a Q or K attention projection.

    Compatible with make_block_builder from Part 9.
    """

    n = embed_dim
    p = head_dim

    if p > n:
        raise ValueError(
            f"Infeasible attention projection: "
            f"p={p} > n={n}"
        )

    # Construct metric B
    B, target_rank = make_B_for_attention(
        n=n,
        p=p,
        device=device,
        dtype=torch.float32,
        target_rank=FIXED_RANK,
    )

    # Initialize diffeomorphism
    diffeo = GeneralizedStiefelDiffeomorphismTorch(
        B=B
    )

    # Layer identifier
    layer_name = (
        f"group{group_index}."
        f"block{block_index}."
        f"{proj_name}_proj"
    )

    # Verify feasibility
    if diffeo.rank < p:
        raise ValueError(
            f"rank(B)={diffeo.rank} < p={p} "
            f"for {layer_name}"
        )

    print(
        f"{layer_name}: "
        f"n={n}, "
        f"p={p}, "
        f"target_rank={target_rank}, "
        f"actual_rank={diffeo.rank}"
    )

    return diffeo


In [17]:

# Create the Transformer block builder
block_builder = make_block_builder(
    diffeo_factory
)

# Create Euclidean patch embedding
patch_embed_layer = make_patch_embed_layer(
    embed_dim=192,
    patch_size=4,
)

# Initialize ViT-Tiny/4
model = ViTTiny4(
    img_size=32,
    patch_size=4,
    depth=12,
    embed_dim=192,
    num_heads=3,
    num_classes=10,
    mlp_ratio=4.0,
    dropout=0.0,
    patch_embed_layer=patch_embed_layer,
    block_builder=block_builder,
).to(device)

criterion = nn.CrossEntropyLoss()

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)


group0.block0.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block0.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block1.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block1.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block2.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block2.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block3.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group0.block3.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block0.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block0.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block1.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block1.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block2.q_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block2.k_proj: n=192, p=64, target_rank=192, actual_rank=192
group1.block3.q_proj: n=192, p=64, target_rank=1

In [35]:

@torch.no_grad()
def check_generalized_stiefel(model):
    errors = []

    for module in model.modules():
        if isinstance(module, ProductGStiefelAttentionProjection):
            W = module.reconstruct_weight()
            B = module.diffeo.B

            gram = W.transpose(-1, -2) @ B @ W
            I = torch.eye(
                W.shape[-1],
                device=W.device,
                dtype=W.dtype
            )

            error = torch.linalg.matrix_norm(
                gram - I, ord="fro"
            )

            errors.extend(error.cpu().tolist())

    print("Number of matrices:", len(errors))
    print("Maximum constraint error:", max(errors))

check_generalized_stiefel(model)


Number of matrices: 72
Maximum constraint error: 2.700184722925769e-06


In [36]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: utils_modify
# ------------------------------------------------------------

import torch


def matrix_norm_one(W, per_head=False):
    """
    Compute the induced matrix 1-norm.

    Supports:
        W: (n, p)             Single matrix
        W: (num_heads, n, p)  Batched matrices

    If per_head=True:
        Return one norm per attention head.

    If per_head=False:
        Return the maximum norm across all heads.
    """

    if W.ndim not in (2, 3):
        raise ValueError(
            "Expected W with shape (n,p) "
            "or (num_heads,n,p)"
        )

    # Sum absolute values along matrix rows
    column_sums = torch.abs(W).sum(dim=-2)

    # Maximum absolute column sum
    norms = column_sums.amax(dim=-1)

    if per_head:
        return norms

    return norms.amax()


In [37]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: gutils_modify
# Supports (r, p) and (num_heads, r, p)
# ------------------------------------------------------------

import torch


# ------------------------------------------------------------
# Basic vector utilities
# ------------------------------------------------------------

def norm(v, dim=-1):
    return v.norm(p=2, dim=dim, keepdim=True)


def unit(v, dim=-1, eps=1e-8):
    vnorm = norm(v, dim)
    return v / (vnorm + eps), vnorm


def xTy(x, y):
    return torch.sum(x * y, dim=-1, keepdim=True)


def clip_by_norm(v, clip_norm):
    v_norm = norm(v)

    scale = torch.clamp(
        clip_norm / v_norm.clamp_min(1e-12),
        max=1.0,
    )

    return v * scale


# ------------------------------------------------------------
# Symmetric and skew-symmetric matrix parts
# ------------------------------------------------------------

def sym_matrix(A):
    return 0.5 * (A + A.transpose(-1, -2))


def skew_matrix(A):
    return 0.5 * (A - A.transpose(-1, -2))


# ------------------------------------------------------------
# Stiefel tangent projection
#
# X, G: (..., r, p)
# Constraint: X^T X = I
# ------------------------------------------------------------

def stiefel_proj_tan(X, G):

    sym = sym_matrix(
        X.transpose(-1, -2) @ G
    )

    return G - X @ sym


# ------------------------------------------------------------
# Stiefel normal projection
# ------------------------------------------------------------

def stiefel_proj_norm(X, G):

    sym = sym_matrix(
        X.transpose(-1, -2) @ G
    )

    return X @ sym


# ------------------------------------------------------------
# Polar retraction
# ------------------------------------------------------------

def polar_retraction(Y):

    U, S, Vh = torch.linalg.svd(
        Y,
        full_matrices=False,
    )

    return U @ Vh


# ------------------------------------------------------------
# QR retraction
# ------------------------------------------------------------

def qr_retraction(Y):

    Q, R = torch.linalg.qr(
        Y,
        mode="reduced",
    )

    d = torch.diagonal(
        R,
        dim1=-2,
        dim2=-1,
    )

    ph = torch.where(
        d < 0,
        -torch.ones_like(d),
        torch.ones_like(d),
    )

    return Q * ph.unsqueeze(-2)


# ------------------------------------------------------------
# Cayley skew-symmetric generator
#
# X: (..., r, p)
# U: tangent direction at X
# W: (..., r, r)
# ------------------------------------------------------------

def cayley_skew(X, U):

    XT = X.transpose(-1, -2)
    UT = U.transpose(-1, -2)

    W = (
        U @ XT
        - X @ UT
        + X @ (UT @ X) @ XT
    )

    return skew_matrix(W)


# ------------------------------------------------------------
# Sequential Cayley iteration
#
# X: (..., r, p)
# W: (..., r, r)
# tan_vec: (..., r, p)
#
# Supports scalar or per-head step sizes
# ------------------------------------------------------------

def Cayley_loop(
    X,
    W,
    tan_vec,
    t,
    num_iter=5,
):

    Y = X + t * tan_vec

    for _ in range(num_iter):

        Y = X + 0.5 * t * (
            W @ (X + Y)
        )

    return Y


# ------------------------------------------------------------
# Check Stiefel orthogonality
# ------------------------------------------------------------

@torch.no_grad()
def check_identity(X):

    r, p = X.shape[-2:]

    I = torch.eye(
        p,
        device=X.device,
        dtype=X.dtype,
    )

    residual = (
        X.transpose(-1, -2) @ X - I
    )

    errors = torch.linalg.matrix_norm(
        residual,
        ord="fro",
    )

    print(f"r={r}, p={p}")
    print("Orthogonality errors:", errors)
    print("Maximum error:", errors.max().item())

    return errors


# ------------------------------------------------------------
# Stiefel vector transport by projection
# ------------------------------------------------------------

def stiefel_transport(X, G):
    return stiefel_proj_tan(X, G)


# ------------------------------------------------------------
# Sphere geometry utilities
# Retained from the original WRN implementation
# ------------------------------------------------------------

def gproj(y, g, normalize=False):

    if normalize:
        y, _ = unit(y)

    yTg = xTy(y, g)

    return g - yTg * y


def gexp(y, h, normalize=False):

    if normalize:
        y, _ = unit(y)
        h = gproj(y, h)

    u, hnorm = unit(h)

    return (
        y * hnorm.cos()
        + u * hnorm.sin()
    )


def gpt2(y, h1, h2, normalize=False):

    if normalize:
        h1 = gproj(y, h1)
        h2 = gproj(y, h2)

    u, unorm = unit(h2)
    uTh1 = xTy(u, h1)

    return h1 - uTh1 * (
        unorm.sin() * y
        + (1 - unorm.cos()) * u
    )


def gpt(y, h, normalize=False):

    if normalize:
        h = gproj(y, h)

    u, unorm = unit(h)

    return (
        u * unorm.cos()
        - y * unorm.sin()
    ) * unorm


In [38]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: SGDG Optimizer
# ------------------------------------------------------------

import torch
import random
from torch.optim.optimizer import Optimizer, required

epsilon = 1e-8


class SGDG(Optimizer):
    """
    Stiefel SGD with Cayley retraction.

    Supports Stiefel parameters:
        X: (r, p)
        X: (num_heads, r, p)

    Constraint:
        X^T X = I

    Also supports ordinary Euclidean SGD updates
    when stiefel=False.
    """

    def __init__(
        self,
        params,
        lr=required,
        momentum=0,
        dampening=0,
        weight_decay=0,
        nesterov=False,
        stiefel=False,
        omega=0,
        grad_clip=None,
    ):

        defaults = dict(
            lr=lr,
            momentum=momentum,
            dampening=dampening,
            weight_decay=weight_decay,
            nesterov=nesterov,
            stiefel=stiefel,
            omega=omega,
            grad_clip=grad_clip,
        )

        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError(
                "Nesterov momentum requires positive "
                "momentum and zero dampening"
            )

        super().__init__(params, defaults)

    def __setstate__(self, state):

        super().__setstate__(state)

        for group in self.param_groups:
            group.setdefault("nesterov", False)

    # --------------------------------------------------------
    # Optimization step
    # --------------------------------------------------------

    @torch.no_grad()
    def step(self, closure=None):

        loss = None

        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:

            lr = group["lr"]
            momentum = group["momentum"]
            stiefel = group["stiefel"]
            weight_decay = group["weight_decay"]
            dampening = group["dampening"]
            nesterov = group["nesterov"]

            for p in group["params"]:

                if p.grad is None:
                    continue

                # --------------------------------------------
                # Stiefel update for product coordinates X
                # --------------------------------------------

                if stiefel:

                    X = p.detach()

                    if X.ndim not in (2, 3):
                        raise ValueError(
                            "Stiefel parameters must have "
                            "shape (r,p) or (heads,r,p)"
                        )

                    r, k = X.shape[-2:]

                    if r < k:
                        raise ValueError(
                            f"Stiefel requires r >= p, "
                            f"got r={r}, p={k}"
                        )

                    # Occasional QR reorthogonalization
                    if random.randint(1, 101) == 1:
                        X = qr_retraction(X)

                    # Euclidean gradient
                    G = p.grad.detach()

                    # Riemannian gradient
                    g = stiefel_proj_tan(X, G)

                    # ----------------------------------------
                    # Momentum buffer
                    # ----------------------------------------

                    state = self.state[p]

                    if "momentum_buffer" not in state:
                        state["momentum_buffer"] = (
                            torch.zeros_like(X)
                        )

                    buffer = state["momentum_buffer"]

                    # Project previous momentum onto
                    # the current tangent space
                    transported_buffer = (
                        stiefel_proj_tan(X, buffer)
                    )

                    # Momentum-adjusted descent direction
                    V = (
                        momentum * transported_buffer
                        - g
                    )

                    # Ensure tangency
                    V = stiefel_proj_tan(X, V)

                    # ----------------------------------------
                    # Cayley generator
                    # ----------------------------------------

                    W = cayley_skew(X, V)

                    # Matrix 1-norm for each head
                    W_norm = matrix_norm_one(
                        W,
                        per_head=True,
                    )

                    # Adaptive step-size bound
                    t = 1.0 / (W_norm + epsilon)

                    alpha = torch.clamp(
                        t,
                        max=lr,
                    )

                    # Broadcast individual head step sizes
                    if X.ndim == 3:
                        alpha = alpha[..., None, None]

                    # ----------------------------------------
                    # Sequential Cayley update
                    # ----------------------------------------

                    X_new = Cayley_loop(
                        X,
                        W,
                        V,
                        alpha,
                        num_iter=5,
                    )

                    # Update the Stiefel coordinate
                    p.copy_(X_new)

                    # Transport momentum to the updated point
                    buffer.copy_(
                        stiefel_proj_tan(
                            X_new,
                            W @ X_new,
                        )
                    )

                # --------------------------------------------
                # Euclidean SGD update
                # --------------------------------------------

                else:

                    d_p = p.grad.detach()

                    if weight_decay != 0:
                        d_p = d_p.add(
                            p,
                            alpha=weight_decay,
                        )

                    if momentum != 0:

                        state = self.state[p]

                        if "momentum_buffer" not in state:

                            state["momentum_buffer"] = (
                                d_p.clone()
                            )

                        else:

                            state["momentum_buffer"].mul_(
                                momentum
                            ).add_(
                                d_p,
                                alpha=1 - dampening,
                            )

                        buf = state["momentum_buffer"]

                        if nesterov:
                            d_p = d_p.add(
                                buf,
                                alpha=momentum,
                            )
                        else:
                            d_p = buf

                    p.add_(
                        d_p,
                        alpha=-lr,
                    )

        return loss


In [40]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Parameter Grouping
# Run AFTER model initialization and BEFORE optimizers
# ------------------------------------------------------------

# Collect generalized Stiefel attention projection layers
constrained_layers = []

for name, module in model.named_modules():
    if isinstance(module, ProductGStiefelAttentionProjection):
        constrained_layers.append((name, module))

# Stiefel coordinates X
stiefel_params = [
    module.X
    for _, module in constrained_layers
]

# Euclidean product coordinates Z
all_z_params = [
    module.Z
    for _, module in constrained_layers
]

# Exclude empty Z tensors (when rank = 192)
product_z_params = [
    p for p in all_z_params
    if p.numel() > 0
]

# Collect remaining Euclidean parameters
product_param_ids = {
    id(p) for p in stiefel_params + all_z_params
}

other_params = [
    p for p in model.parameters()
    if p.requires_grad
    and id(p) not in product_param_ids
]

# Final Euclidean parameter list
euclid_params = product_z_params + other_params

# Verify disjoint parameter groups
assert not (
    {id(p) for p in stiefel_params}
    & {id(p) for p in euclid_params}
)

# Verify all nonempty trainable parameters are covered
assert (
    {id(p) for p in stiefel_params + euclid_params}
    ==
    {
        id(p) for p in model.parameters()
        if p.requires_grad and p.numel() > 0
    }
)

# Summary
print("Constrained projection modules:", len(constrained_layers))
print("Stiefel parameter tensors:", len(stiefel_params))
print("Individual Stiefel matrices:",
      sum(p.shape[0] for p in stiefel_params))
print("Nonempty Z tensors:", len(product_z_params))
print("Euclidean parameter tensors:", len(euclid_params))


Constrained projection modules: 24
Stiefel parameter tensors: 24
Individual Stiefel matrices: 72
Nonempty Z tensors: 0
Euclidean parameter tensors: 176


In [41]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Optimizers and Learning-Rate Schedulers
# ------------------------------------------------------------

LR_STIEFEL = 0.2
LR_EUCLID = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# ------------------------------------------------------------
# Learning-rate scheduler settings
# ------------------------------------------------------------

LR_MILESTONES = [12, 60, 120, 160]
LR_GAMMA = 0.2

# ------------------------------------------------------------
# Stiefel optimizer: Product coordinates X
# ------------------------------------------------------------

stiefel_optimizer = SGDG(
    [
        {
            "params": stiefel_params,
            "lr": LR_STIEFEL,
            "stiefel": True,
            "momentum": 0.0,
            "weight_decay": WEIGHT_DECAY,
        }
    ]
)

# ------------------------------------------------------------
# Euclidean optimizer:
# Product coordinates Z + remaining parameters
# ------------------------------------------------------------

euclid_optimizer = torch.optim.SGD(
    euclid_params,
    lr=LR_EUCLID,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# ------------------------------------------------------------
# Stiefel learning-rate scheduler
# ------------------------------------------------------------

stiefel_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    stiefel_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

# ------------------------------------------------------------
# Euclidean learning-rate scheduler
# ------------------------------------------------------------

euclid_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    euclid_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

# ------------------------------------------------------------
# Optimizer summary
# ------------------------------------------------------------

print("stiefel_optimizer =", stiefel_optimizer)
print("euclid_optimizer =", euclid_optimizer)

print("Stiefel learning rate:", LR_STIEFEL)
print("Euclidean learning rate:", LR_EUCLID)
print("Euclidean momentum:", MOMENTUM)
print("Weight decay:", WEIGHT_DECAY)
print("Fixed metric rank:", FIXED_RANK)


stiefel_optimizer = SGDG (
Parameter Group 0
    dampening: 0
    grad_clip: None
    initial_lr: 0.2
    lr: 0.2
    momentum: 0.0
    nesterov: False
    omega: 0
    stiefel: True
    weight_decay: 0.0005
)
euclid_optimizer = SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.01
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0005
)
Stiefel learning rate: 0.2
Euclidean learning rate: 0.01
Euclidean momentum: 0.9
Weight decay: 0.0005
Fixed metric rank: 192


In [42]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Loss Function and Training Configuration
# ------------------------------------------------------------

# Cross-entropy loss for CIFAR-10 classification
criterion = nn.CrossEntropyLoss()

# Number of training epochs
num_epochs = 200

# ------------------------------------------------------------
# Training configuration summary
# ------------------------------------------------------------

print("Starting ViT-Tiny/4 Fixed-Rank training...")
print("Device:", device)
print("Number of epochs:", num_epochs)
print("Batch size:", batch_size)
print("Fixed rank:", FIXED_RANK)


Starting ViT-Tiny/4 Fixed-Rank training...
Device: cpu
Number of epochs: 200
Batch size: 128
Fixed rank: 192


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Training and Validation Loop
# ------------------------------------------------------------

# Training metrics
train_losses = []
train_accuracies = []

# Validation metrics
val_losses = []
val_accuracies = []

# Model was already moved to device during initialization
# and before the optimizers were constructed.

# ------------------------------------------------------------
# Outer training loop
# ------------------------------------------------------------

for epoch in range(num_epochs):

    model.train()

    # Training statistics
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    # --------------------------------------------------------
    # Training phase
    # --------------------------------------------------------

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients
        stiefel_optimizer.zero_grad()
        euclid_optimizer.zero_grad()

        # Forward pass through ViT-Tiny/4
        # Q/K weights are reconstructed from X and Z
        outputs = model(images)

        # Cross-entropy loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # SGDG: update Stiefel product coordinates X
        stiefel_optimizer.step()

        # SGD: update Z and other Euclidean parameters
        euclid_optimizer.step()

        # Accumulate training statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, dim=1)

        running_correct += (
            predicted == labels
        ).sum().item()

        running_total += labels.size(0)

    # --------------------------------------------------------
    # Average training metrics
    # --------------------------------------------------------

    epoch_train_loss = running_loss / running_total
    epoch_train_acc = running_correct / running_total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # --------------------------------------------------------
    # Validation phase
    # --------------------------------------------------------

    model.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    val_running_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Validation loss
            loss = criterion(outputs, labels)

            val_running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(outputs, dim=1)

            val_running_correct += (
                predicted == labels
            ).sum().item()

            val_running_total += labels.size(0)

    # --------------------------------------------------------
    # Average validation metrics
    # --------------------------------------------------------

    epoch_val_loss = val_running_loss / val_running_total
    epoch_val_acc = val_running_correct / val_running_total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # --------------------------------------------------------
    # Learning-rate scheduling
    # --------------------------------------------------------

    # Learning rates used during this epoch
    current_lr_stiefel = (
        stiefel_optimizer.param_groups[0]["lr"]
    )

    current_lr_euclid = (
        euclid_optimizer.param_groups[0]["lr"]
    )

    # Update learning rates for the next epoch
    stiefel_scheduler.step()
    euclid_scheduler.step()

    # --------------------------------------------------------
    # Print epoch results
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Rank: {FIXED_RANK} | "
        f"LR_Euclid: {current_lr_euclid:.6f} | "
        f"LR_Stiefel: {current_lr_stiefel:.6f} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f}"
    )


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Test Evaluation
# ------------------------------------------------------------

model.eval()

# Initialize test statistics
test_running_loss = 0.0
test_running_correct = 0
test_running_total = 0

# CIFAR-10 class names
classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

# Per-class statistics
class_correct = {cls_name: 0 for cls_name in classes}
class_total = {cls_name: 0 for cls_name in classes}

# ------------------------------------------------------------
# Test loop
# ------------------------------------------------------------

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # ViT-Tiny/4 forward pass
        # Q/K weights are reconstructed using phi_inverse
        outputs = model(images)

        # Cross-entropy loss
        loss = criterion(outputs, labels)

        # Accumulate test loss
        test_running_loss += (
            loss.item() * images.size(0)
        )

        # Predicted class indices
        _, predicted = torch.max(outputs, dim=1)

        # Overall accuracy statistics
        test_running_correct += (
            predicted == labels
        ).sum().item()

        test_running_total += labels.size(0)

        # Per-class accuracy statistics
        for true_label, pred_label in zip(labels, predicted):

            true_idx = true_label.item()
            cls_name = classes[true_idx]

            class_total[cls_name] += 1

            if pred_label.item() == true_idx:
                class_correct[cls_name] += 1

# ------------------------------------------------------------
# Overall test metrics
# ------------------------------------------------------------

test_loss = test_running_loss / test_running_total
test_accuracy = test_running_correct / test_running_total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# ------------------------------------------------------------
# Per-class accuracy
# ------------------------------------------------------------

class_accuracies = {}

print("\nPer-class accuracy:")

for cls_name in classes:

    if class_total[cls_name] > 0:

        acc = (
            class_correct[cls_name]
            / class_total[cls_name]
        )

        class_accuracies[cls_name] = acc

        print(f"{cls_name:>5s}: {acc:.4f}")


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank:
# Save Training Metrics and Checkpoint
# ------------------------------------------------------------

import os
import torch
import pandas as pd

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

output_dir = "vit_fixed_rank"
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# Save training metrics
# ------------------------------------------------------------

results = pd.DataFrame({
    "epoch": list(range(1, len(train_losses) + 1)),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "val_loss": val_losses,
    "val_accuracy": val_accuracies,
})

metrics_path = os.path.join(
    output_dir,
    "training_metrics.csv"
)

results.to_csv(metrics_path, index=False)

# ------------------------------------------------------------
# Save model checkpoint
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    output_dir,
    f"vit_tiny4_fixed_rank_{FIXED_RANK}_checkpoint.pt"
)

torch.save(
    {
        # Model configuration
        "model_name": "vit_tiny4",
        "dataset": "CIFAR10",
        "img_size": 32,
        "patch_size": 4,
        "depth": 12,
        "embed_dim": 192,
        "num_heads": 3,
        "num_classes": 10,
        "mlp_ratio": 4.0,
        "dropout": 0.0,

        # Fixed-rank configuration
        "fixed_rank": FIXED_RANK,
        "constraint": "generalized_stiefel",
        "constrained_projections": ["q", "k"],

        # Training configuration
        "num_epochs": num_epochs,
        "batch_size": batch_size,
        "lr_stiefel": LR_STIEFEL,
        "lr_euclid": LR_EUCLID,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "lr_milestones": LR_MILESTONES,
        "lr_gamma": LR_GAMMA,

        # Model parameters and metric buffers
        "model_state_dict": model.state_dict(),

        # Optimizer states
        "stiefel_optimizer_state_dict":
            stiefel_optimizer.state_dict(),

        "euclid_optimizer_state_dict":
            euclid_optimizer.state_dict(),

        # Scheduler states
        "stiefel_scheduler_state_dict":
            stiefel_scheduler.state_dict(),

        "euclid_scheduler_state_dict":
            euclid_scheduler.state_dict(),

        # Training history
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,

        # Test results
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "class_accuracies": class_accuracies,
    },
    checkpoint_path,
)

print(f"Saved {metrics_path}")
print(f"Saved {checkpoint_path}")


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Fixed Rank: Training Plots
# ------------------------------------------------------------

import os
import matplotlib.pyplot as plt

output_dir = "vit_fixed_rank"
os.makedirs(output_dir, exist_ok=True)

epochs = range(1, len(train_losses) + 1)

# ------------------------------------------------------------
# Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    train_losses,
    marker="o",
    label="Train Loss",
)

plt.plot(
    epochs,
    val_losses,
    marker="s",
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    f"ViT-Tiny/4 Fixed Rank {FIXED_RANK}: "
    "Training and Validation Loss"
)
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "loss_curves.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.close()

# ------------------------------------------------------------
# Training and validation accuracy
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    [acc * 100 for acc in train_accuracies],
    marker="o",
    label="Train Accuracy",
)

plt.plot(
    epochs,
    [acc * 100 for acc in val_accuracies],
    marker="s",
    label="Validation Accuracy",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title(
    f"ViT-Tiny/4 Fixed Rank {FIXED_RANK}: "
    "Training and Validation Accuracy"
)
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "accuracy_curves.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print("Saved:", os.path.join(output_dir, "loss_curves.png"))
print("Saved:", os.path.join(output_dir, "accuracy_curves.png"))
